# 🖥️ Datasets with Time Varying Z Coordinates

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation

import parcels
from parcels._datasets.structured.generated import (
    simple_UV_dataset,
    simple_UV_sigma_dataset,
)
from parcels._datasets.unstructured.generated import sigma_coordinate_lattice_dataset

plt.rcParams["animation.html"] = "jshtml"

## Building the Dataset

For this tutorial, we construct our dataset using the parcels built in `simple_UV_sigma_dataset`, which makes a structured horizontally gridded dataset that uses a linear vertical $\sigma$ grid. Its horizontal grid is a flat square that spans from -1000 km to 1000 km in both $x$ and $y$. Here, we assume $\frac{\partial \sigma}{\partial z}$ to be constant, so for any given z coordinate, the equivalent $\sigma$ coordinate is a dimensionless fraction of column height, i.e. $\sigma = \frac{z + \eta}{H +\eta}$, where $H$ is the depth of the water column, and $\eta$ is the sea surface height. It is common in $\sigma$ coordinate models to use more advanced vertical grids for which $\frac{\partial \sigma}{\partial z}$ is not constant, this allows the concentration of vertical layers near the surface. Nonetheless, our simple example allows for a vertical coordinate system that adapts to changes in bathymetry and variations in sea surface height. We will demonstrate this shortly but begin by constructing the dataset itself. At the end of the tutorial, we show the same setup on an unstructured grid, and how to modify the interpolation method used on the multi-dimensional z coordinate.

In [ ]:
n_days = 10
n_snapshots = 24 * n_days + 1  # hourly snapshots from hour 0 to the end of day 10
n_vertical_layers = 10
ny, nx = (
    5,
    81,
)  # the number of grid nodes in y and x, so that the cells are 25km wide in x

lon = simple_UV_dataset(
    n_times=n_snapshots, nz=n_vertical_layers + 1, ny=ny, nx=nx, mesh_str="flat"
)["lon"].values
dx = lon[1] - lon[0]
x_centres = lon - 0.5 * dx

# we construct a depth profile that is 100m deep in the west and logarithmically increases to 1000m depth in the east
domain_length = lon[-1] - lon[0]
bottom_depth = 100 + 900 * np.log10(1 + 9 * (x_centres - lon[0]) / domain_length)
bottom_depth = np.broadcast_to(bottom_depth, (ny, nx))

# for illustrative purposes we create a 50m surface wave with a 1000km wavelength that propagates east with time
t = np.arange(n_snapshots) * 3600.0
wavelength, wave_period = 1000e3, 2 * 86400.0
amplitude = 50
wave = amplitude * np.sin(
    2.0
    * np.pi
    * (x_centres[np.newaxis, :] / wavelength - t[:, np.newaxis] / wave_period)
)
wave = np.broadcast_to(wave[:, np.newaxis, :], (n_snapshots, ny, nx))

# construct the dataset itself
ds = simple_UV_sigma_dataset(
    n_times=n_snapshots,
    nz=n_vertical_layers + 1,
    ny=ny,
    nx=nx,
    bottom_depth=bottom_depth,
    eta=wave,
)
ds

We can now compare the evolution of the vertical grid in $\sigma$ space to the vertical grid in z space. In z space, the vertical grid spacing is much smaller in the shallow west than in the deep east, this demonstrates the strength of a terrain following vertical grid. We can also see that the vertical grid has time dependency in z space. The z coordinate is defined at the cell centres, and by default Parcels interpolates time-varying z bilinearly in space between the surrounding cell centres, so along a row of cells the layer interfaces are straight lines between the cell centres.

In [ ]:
time = ds["time"].values.astype("datetime64[s]")
z_moving = ds["depth"].values  # interface depths, shape (time, depth, YC, XC)
sigma = np.linspace(0.0, 1.0, n_vertical_layers + 1)

centre_row = ny // 2  # the row of cells in the middle of the domain
xs = x_centres / 1e3

fig, (ax_sigma, ax_z) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
ax_sigma.plot(xs, np.broadcast_to(sigma, (xs.size, sigma.size)), "k", linewidth=0.8)
ax_sigma.set_ylabel(r"$\sigma$")
ax_sigma.set_title(r"Layer interfaces in $\sigma$ space")

z_lines = ax_z.plot(xs, z_moving[0][:, centre_row].T, "k", linewidth=0.8)
ax_z.set_ylabel("depth [m]")
z_title = ax_z.set_title(f"Layer interfaces in z space at {time[0]}")

for ax in (ax_sigma, ax_z):
    ax.invert_yaxis()
    ax.set_xlabel("x [km]")
plt.tight_layout()


def animate(snapshot):
    for line, interface_depths in zip(
        z_lines, z_moving[snapshot][:, centre_row], strict=True
    ):
        line.set_ydata(interface_depths)
    z_title.set_text(f"Layer interfaces in z space at {time[snapshot]}")


anim = FuncAnimation(fig, animate, frames=range(0, time.size, 6), interval=300)
plt.close()
anim

## Building the `FieldSet`

We prescribe a uniform velocity of $u_0 = 1$ m/s in the $x$ direction, with $v = w = 0$. All three components sit on the grid nodes at the layer interfaces (`depth`), so Parcels interpolates them linearly in all three directions.

To advect the particles on a sigma grid, for which the z grid varies in time, we use a 4D `depth` coordinate with dimensions of `(time, depth, YC, XC)`. For each particle, Parcels builds a z column by interpolating the columns at the surrounding cell centres bilinearly in space and linearly in time between the two surrounding snapshots, and then searches that column for the particle's layer.

In [ ]:
u0 = 1.0
ds["U"].values[:] = u0
ds

In [ ]:
fieldset = parcels.FieldSet.from_sgrid_conventions(ds)
fieldset.describe()

## Release the particles

We release 10 particles at the first snapshot near the shallow western end of the domain, at $x = -890$ km and $y = 0$, at depths from 60 m to 240 m. This puts the shallowest particle just below the deepest wave trough (50 m), so every particle will remain in the water column throughout the simulation.

In [ ]:
n_particles = 10
x0 = np.full(n_particles, -890e3)
y0 = np.zeros(n_particles)
z0 = np.linspace(60.0, 240.0, n_particles)  # depths in m

pset = parcels.ParticleSet(
    fieldset=fieldset,
    pclass=parcels.Particle,
    t=np.full(n_particles, time[0]),
    z=z0,
    y=y0,
    x=x0,
)

In [ ]:
H = z_moving[0, -1, centre_row]
eta = -z_moving[:, 0, centre_row]
# initial sigma coordinate of each particle, using its z column interpolated linearly in x between the cell centres
eta0 = np.interp(x0, x_centres, eta[0])
sigma0 = (z0 + eta0) / (np.interp(x0, x_centres, H) + eta0)

colors = [f"C{p}" for p in range(n_particles)]

fig, (ax_sigma, ax_z) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
ax_sigma.plot(xs, np.broadcast_to(sigma, (xs.size, sigma.size)), "k", linewidth=0.8)
ax_sigma.scatter(x0 / 1e3, sigma0, color=colors, zorder=3)
ax_sigma.set_ylabel(r"$\sigma$")
ax_sigma.set_title(r"Initial particle positions in $\sigma$ space")

ax_z.plot(xs, z_moving[0][:, centre_row].T, "k", linewidth=0.8)
ax_z.scatter(x0 / 1e3, z0, color=colors, zorder=3)
ax_z.set_ylabel("depth [m]")
ax_z.set_title(f"Initial particle positions in z space at {time[0]}")

for ax in (ax_sigma, ax_z):
    ax.invert_yaxis()
    ax.set_xlabel("x [km]")
plt.tight_layout()
plt.show()

## Advect the particles

We advect the particles for 10 days using the 2D second-order Runge-Kutta kernel {py:func}`parcels.kernels.AdvectionRK2` with a 30-minute time step, and write their positions every 6 hours. We chose the 2D version of RK2 since the vertical velocity is zero across the domain.

In [ ]:
outputfile = parcels.ParticleFile(
    path="sigma_particles.parquet", outputdt=np.timedelta64(6, "h"), mode="w"
)

pset.execute(
    [parcels.kernels.AdvectionRK2],
    runtime=np.timedelta64(n_days, "D"),
    dt=np.timedelta64(30, "m"),
    output_file=outputfile,
    verbose_progress=False,
)

In [ ]:
df = parcels.read_particlefile("sigma_particles.parquet").sort(["t", "particle_id"])
output_times = df["t"].unique().sort()
n_frames = len(output_times)
px = df["x"].to_numpy().reshape(n_frames, n_particles)
pz = df["z"].to_numpy().reshape(n_frames, n_particles)

# calculate the sigma coordinate of every particle at every output time, using its z column interpolated linearly in x
# between the cell centres
snapshots = np.searchsorted(time, output_times.to_numpy(), side="right") - 1
psigma = np.empty_like(pz)
for frame, snapshot in enumerate(snapshots):
    eta_p = np.interp(px[frame], x_centres, eta[snapshot])
    psigma[frame] = (pz[frame] + eta_p) / (np.interp(px[frame], x_centres, H) + eta_p)

colors = [f"C{p}" for p in range(n_particles)]

fig, (ax_sigma, ax_z) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
ax_sigma.plot(xs, np.broadcast_to(sigma, (xs.size, sigma.size)), "k", linewidth=0.8)
z_lines = ax_z.plot(xs, z_moving[0][:, centre_row].T, "k", linewidth=0.8)
sigma_trails = [ax_sigma.plot([], [], color=c, linewidth=1.5)[0] for c in colors]
z_trails = [ax_z.plot([], [], color=c, linewidth=1.5)[0] for c in colors]
sigma_dots = ax_sigma.scatter(px[0] / 1e3, psigma[0], c=colors, zorder=3)
z_dots = ax_z.scatter(px[0] / 1e3, pz[0], c=colors, zorder=3)

ax_sigma.set_ylabel(r"$\sigma$")
ax_sigma.set_title(r"Particles in $\sigma$ space")
ax_z.set_ylabel("depth [m]")
z_title = ax_z.set_title("")
for ax in (ax_sigma, ax_z):
    ax.invert_yaxis()
    ax.set_xlabel("x [km]")
plt.tight_layout()


def animate(frame):
    for p in range(n_particles):
        sigma_trails[p].set_data(px[: frame + 1, p] / 1e3, psigma[: frame + 1, p])
        z_trails[p].set_data(px[: frame + 1, p] / 1e3, pz[: frame + 1, p])
    sigma_dots.set_offsets(np.c_[px[frame] / 1e3, psigma[frame]])
    z_dots.set_offsets(np.c_[px[frame] / 1e3, pz[frame]])
    for line, interface_depths in zip(
        z_lines, z_moving[snapshots[frame]][:, centre_row], strict=True
    ):
        line.set_ydata(interface_depths)
    z_title.set_text(f"Particles in z space at {output_times[frame]}")


animate(0)
anim = FuncAnimation(fig, animate, frames=n_frames, interval=300)
plt.close()
anim

We can see that, as expected, the particles move horizontally in x and cross the terrain following depth levels. However, in z space, they maintain their exact height as expected.

## Sigma coordinates on unstructured grids

The same approach works on unstructured grids. On an unstructured grid, z is a 3D `zf` coordinate with dimensions `(time, zf, n_node/n_face)`, rather than a 4D `depth` coordinate with dimensions `(time, depth, Y, X)`. 

Below we build the same basin on a triangulated lattice with `sigma_coordinate_lattice_dataset`, evaluating the bathymetry and sea surface height at the lattice nodes, and compare the two z coordinates.

In [ ]:
n_lattice_nodes = (
    21  # the number of lattice nodes in x and y, so that the nodes are 100km apart
)
x_nodes = np.linspace(-1000e3, 1000e3, n_lattice_nodes)

# the same bathymetry and surface wave as above, evaluated at the lattice nodes, with shapes (x, y) and (time, x, y)
bottom_depth_nodes = 100 + 900 * np.log10(
    1 + 9 * (x_nodes - x_nodes[0]) / domain_length
)
bottom_depth_nodes = np.broadcast_to(
    bottom_depth_nodes[:, np.newaxis], (n_lattice_nodes, n_lattice_nodes)
)
wave_nodes = amplitude * np.sin(
    2.0 * np.pi * (x_nodes[np.newaxis, :] / wavelength - t[:, np.newaxis] / wave_period)
)
wave_nodes = np.broadcast_to(
    wave_nodes[:, :, np.newaxis], (n_snapshots, n_lattice_nodes, n_lattice_nodes)
)

ux_ds = sigma_coordinate_lattice_dataset(
    n_lattice_nodes,
    (-1000e3, 1000e3),
    (-1000e3, 1000e3),
    n_vertical_layers + 1,
    bottom_depth_nodes,
    eta=wave_nodes,
)
print("structured z:  ", ds["depth"].dims, ds["depth"].shape)
print("unstructured z:", ux_ds["zf"].dims, ux_ds["zf"].shape)

In [ ]:
ux_ds["U"].values[:] = u0
ux_fieldset = parcels.FieldSet.from_ugrid_conventions(ux_ds)
ux_fieldset.describe()

We release the same 10 particles at the same positions on the unstructured grid, and advect them with the same kernel, time step and output frequency as before.

In [ ]:
ux_pset = parcels.ParticleSet(
    fieldset=ux_fieldset,
    pclass=parcels.Particle,
    t=np.full(n_particles, time[0]),
    z=z0,
    y=y0,
    x=x0,
)
ux_outputfile = parcels.ParticleFile(
    path="ux_sigma_particles.parquet", outputdt=np.timedelta64(6, "h"), mode="w"
)

ux_pset.execute(
    [parcels.kernels.AdvectionRK2],
    runtime=np.timedelta64(n_days, "D"),
    dt=np.timedelta64(30, "m"),
    output_file=ux_outputfile,
    verbose_progress=False,
)

In [ ]:
ux_df = parcels.read_particlefile("ux_sigma_particles.parquet").sort(
    ["t", "particle_id"]
)
ux_px = ux_df["x"].to_numpy().reshape(n_frames, n_particles)
ux_pz = ux_df["z"].to_numpy().reshape(n_frames, n_particles)

# the particles move along the row of lattice nodes at y = 0, i.e. along triangle edges, so the z column that
# Parcels uses for each particle is the linear interpolation in x of the z columns at the nodes of that row
row = np.isclose(ux_ds.uxgrid.node_lat.values, 0.0)
z_nodes = ux_ds["zf"].values[
    :, :, row
]  # interface depths along y = 0, shape (time, zf, x)
H_nodes, eta_nodes = z_nodes[0, -1], -z_nodes[:, 0]
ux_psigma = np.empty_like(ux_pz)
for frame, snapshot in enumerate(snapshots):
    ux_eta_p = np.interp(ux_px[frame], x_nodes, eta_nodes[snapshot])
    ux_psigma[frame] = (ux_pz[frame] + ux_eta_p) / (
        np.interp(ux_px[frame], x_nodes, H_nodes) + ux_eta_p
    )

fig, (ax_sigma, ax_z) = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
ax_sigma.plot(
    x_nodes / 1e3,
    np.broadcast_to(sigma, (x_nodes.size, sigma.size)),
    "k",
    linewidth=0.8,
)
z_lines = ax_z.plot(x_nodes / 1e3, z_nodes[0].T, "k", linewidth=0.8)
sigma_trails = [ax_sigma.plot([], [], color=c, linewidth=1.5)[0] for c in colors]
z_trails = [ax_z.plot([], [], color=c, linewidth=1.5)[0] for c in colors]
sigma_dots = ax_sigma.scatter(ux_px[0] / 1e3, ux_psigma[0], c=colors, zorder=3)
z_dots = ax_z.scatter(ux_px[0] / 1e3, ux_pz[0], c=colors, zorder=3)

ax_sigma.set_ylabel(r"$\sigma$")
ax_sigma.set_title(r"Particles in $\sigma$ space on the unstructured grid")
ax_z.set_ylabel("depth [m]")
z_title = ax_z.set_title("")
for ax in (ax_sigma, ax_z):
    ax.invert_yaxis()
    ax.set_xlabel("x [km]")
plt.tight_layout()


def animate(frame):
    for p in range(n_particles):
        sigma_trails[p].set_data(ux_px[: frame + 1, p] / 1e3, ux_psigma[: frame + 1, p])
        z_trails[p].set_data(ux_px[: frame + 1, p] / 1e3, ux_pz[: frame + 1, p])
    sigma_dots.set_offsets(np.c_[ux_px[frame] / 1e3, ux_psigma[frame]])
    z_dots.set_offsets(np.c_[ux_px[frame] / 1e3, ux_pz[frame]])
    for line, interface_depths in zip(z_lines, z_nodes[snapshots[frame]]):
        line.set_ydata(interface_depths)
    z_title.set_text(f"Particles in z space at {output_times[frame]}")


animate(0)
anim = FuncAnimation(fig, animate, frames=n_frames, interval=300)
plt.close()
anim

As on the structured grid, the particles keep their exact height in z space while crossing the terrain following depth levels.

## Choosing how z is interpolated

With a 4d z coordinate, in order to find a particle z value at some time, we must interpolate the current z grid at that time onto the particle position. This reduces the 4d z coordinate to a 1d z coordiante where each entry is an interface depth at the particle's exact position in space. For more details on interpolation in general, see [tutorial_interpolation](tutorial_interpolation.ipynb). There are currently a number of built in interpolators for vertical coordinates:

- `XVerticalCoordinateLinear` (structured grids, z on the cell centres or corners): bilinear interpolation between the surrounding columns, so z is continuous in space.
- `XVerticalCoordinateNearest` (structured grids, z on the cell centres or corners): piecewise constant interpolation based on whatever cell is closest to the particle.
- `XVerticalCoordinateConstant` (structured grids, z on the cell centres): piecewise constant interpolation based on whatever cell the particle is in. 
- `UxVerticalCoordinateLinearNode` (unstructured grids, z on the nodes): barycentric interpolation from the three corners of the particle's triangle.
- `UxVerticalCoordinateConstantFace` (unstructured grids, z on the faces): piecewise constant interpolation based on whatever cell the particle is in. 

All of them interpolate z linearly in time between the two surrounding snapshots.

The effect of selecting different vertical coordinate interpolators can be seen by investigating what happens to particles near the surface:

In [ ]:
# the same basin as above on a coarser grid with 200 km wide cells, so that the difference between the interpolators is
# easy to see
coarse_nx = 11
coarse_lon = np.linspace(lon[0], lon[-1], coarse_nx)
coarse_x_centres = coarse_lon - 0.5 * (coarse_lon[1] - coarse_lon[0])
# the first cell centre lies west of the domain, where the depth profile is held at its western value of 100 m
coarse_bottom_depth = 100 + 900 * np.log10(
    1 + 9 * np.clip(coarse_x_centres - lon[0], 0.0, None) / domain_length
)
coarse_wave = amplitude * np.sin(
    2.0
    * np.pi
    * (coarse_x_centres[np.newaxis, :] / wavelength - t[:, np.newaxis] / wave_period)
)
coarse_ds = simple_UV_sigma_dataset(
    n_times=n_snapshots,
    nz=n_vertical_layers + 1,
    ny=ny,
    nx=coarse_nx,
    bottom_depth=np.broadcast_to(coarse_bottom_depth, (ny, coarse_nx)),
    eta=np.broadcast_to(coarse_wave[:, np.newaxis, :], (n_snapshots, ny, coarse_nx)),
)
coarse_ds["U"].values[:] = u0
coarse_fieldset = parcels.FieldSet.from_sgrid_conventions(coarse_ds)


def DeleteParticle(particles, fieldset):
    any_error = particles.state >= 50  # e.g. particles that are above the sea surface
    particles[any_error].state = parcels.StatusCode.Delete


# release particles near the surface, 15 km west of the cell edge at x = -400 km, where the sea surface of the next
# cell is lower than in the cell they start in, and advect them for one day with each interpolator
surface_z0 = np.linspace(22.0, 31.0, 4)  # depths in m
z_interp_methods = {
    "XVerticalCoordinateLinear": parcels.interpolators.XVerticalCoordinateLinear(),
    "XVerticalCoordinateConstant": parcels.interpolators.XVerticalCoordinateConstant(),
}
for name, z_interp_method in z_interp_methods.items():
    coarse_fieldset.U.grid.z_interp_method = z_interp_method
    surface_pset = parcels.ParticleSet(
        fieldset=coarse_fieldset,
        pclass=parcels.Particle,
        t=np.full(surface_z0.size, time[0]),
        z=surface_z0,
        y=np.zeros(surface_z0.size),
        x=np.full(surface_z0.size, -415e3),
    )
    surface_outputfile = parcels.ParticleFile(
        path=f"{name}_particles.parquet", outputdt=np.timedelta64(1, "h"), mode="w"
    )
    surface_pset.execute(
        [parcels.kernels.AdvectionRK2, DeleteParticle],
        runtime=np.timedelta64(1, "D"),
        dt=np.timedelta64(30, "m"),
        output_file=surface_outputfile,
        verbose_progress=False,
    )

In [ ]:
n_surface_frames = 25  # hourly outputs from hour 0 to hour 24
surface_px = {}
for name in z_interp_methods:
    surface_df = parcels.read_particlefile(f"{name}_particles.parquet")
    frames = (surface_df["t"].to_numpy() - time[0]) // np.timedelta64(1, "h")
    surface_px[name] = np.full((n_surface_frames, surface_z0.size), np.nan)
    surface_px[name][frames, surface_df["particle_id"].to_numpy()] = surface_df["x"]
    # a particle that has left the water stays at its last position
    for frame in range(1, n_surface_frames):
        left_the_water = np.isnan(surface_px[name][frame])
        surface_px[name][frame, left_the_water] = surface_px[name][frame - 1, left_the_water]

coarse_z = coarse_ds["depth"].values[:, :, centre_row]  # interface depths, shape (time, depth, x)
surface_colors = [f"C{p}" for p in range(surface_z0.size)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
panels = []
for ax, name, drawstyle in zip(
    axes, z_interp_methods, ["default", "steps-mid"], strict=True
):
    lines = ax.plot(
        coarse_x_centres / 1e3, coarse_z[0].T, "k", linewidth=0.8, drawstyle=drawstyle
    )
    dots = ax.scatter(surface_px[name][0] / 1e3, surface_z0, c=surface_colors, zorder=3)
    ax.set_title(name)
    ax.set_xlim(-800, 0)
    ax.set_xlabel("x [km]")
    panels.append((name, lines, dots))
axes[0].set_ylim(300, -75)  # the upper part of the water column, with depth increasing downwards
axes[0].set_ylabel("depth [m]")
suptitle = fig.suptitle(f"Particles near the surface at {time[0]}")
plt.tight_layout()


def animate(frame):
    for name, lines, dots in panels:
        for line, interface_depths in zip(lines, coarse_z[frame], strict=True):
            line.set_ydata(interface_depths)
        dots.set_offsets(np.c_[surface_px[name][frame] / 1e3, surface_z0])
    suptitle.set_text(f"Particles near the surface at {time[frame]}")


animate(0)
anim = FuncAnimation(fig, animate, frames=n_surface_frames, interval=300)
plt.close()
anim

With `XVerticalCoordinateLinear`, the sea surface and the layer interfaces vary smoothly between the cell centres, while with `XVerticalCoordinateConstant` they jump at every cell edge. The discontinuity of piecewise constant vertical coordinate interpolators can cause particles to spuriously leave the domain, as shown above. 

By default, Parcels uses `XVerticalCoordinateLinear` for time-varying z on structured grids, `UxVerticalCoordinateLinearNode` for z on the nodes of an unstructured grid, and `UxVerticalCoordinateConstantFace` for z on the faces of an unstructured grid. To use a different interpolator, set the `z_interp_method` of the grid after building the `FieldSet`, which changes it for every field on that grid:

```python
fieldset.U.grid.z_interp_method = parcels.interpolators.XVerticalCoordinateConstant()
```

Parcels raises an error when the interpolator does not fit where z is defined, for example `XVerticalCoordinateConstant` with z on the cell corners.